In [ ]:
# Install required packages and load environment variables
%pip install anthropic
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()
model = "claude-sonnet-4-6"

In [ ]:
# Helper functions — chat() now supports stop_sequences
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# Tanpa teknik apapun — Claude jawab dengan markdown formatting
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")

text = chat(messages)
text

In [ ]:
# Teknik 1: Prefill assistant message
# Dengan menambahkan pesan assistant yang dimulai dengan ```json,
# Claude akan melanjutkan dari titik itu — langsung isi JSON-nya
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")  # <-- prefill

text = chat(messages)
text

In [ ]:
# Teknik 2: Prefill + Stop Sequence
# stop_sequences=["```"] membuat Claude berhenti saat menemukan penutup ```
# Hasilnya: JSON bersih tanpa markdown wrapper sama sekali
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")  # <-- prefill

text = chat(messages, stop_sequences=["```"])  # <-- stop sebelum penutup ```
text

In [ ]:
# Hasilnya sekarang bisa langsung di-parse sebagai JSON
import json

messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
data = json.loads(text.strip())
print(json.dumps(data, indent=2))